Notebook 1  Setup & API config

Phase 1  Install libraries


In [ ]:
%pip install -r requirements.txt

In [ ]:
%pip install langchain langchain-google-genai groq newsapi-python streamlit python-dotenv


Phase 2  Test the config

In [ ]:
import sys
print(sys.executable)

## Phase 1  Verify API Keys from .env

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

keys = {
    "GROQ_API_KEY": os.getenv("GROQ_API_KEY"),
    "NEWSAPI_KEY": os.getenv("NEWSAPI_KEY"),
}

print("=== API KEY STATUS ===")
for name, val in keys.items():
    if val:
        print(f"{name} = {val}")
    else:
        print(f"{name} NOT FOUND  check your .env file")


NewsAPI test

In [ ]:
from newsapi import NewsApiClient

Fetch news test

In [ ]:
newsapi = NewsApiClient(api_key=os.getenv("NEWSAPI_KEY"))

In [ ]:
articles = newsapi.get_everything(
    q="AI",
    language="en",
    page_size=3
)

## Phase 2  Import and Test langchain_config.py

In [ ]:
#  Import the config file
import sys
sys.path.append('.')  # tells notebook to look in the same folder

try:
    from langchain_config import (
        analyze_news,
        get_summary,
        get_news_articles,
        summarize_articles,
        get_chain
    )

    print("langchain_config imported successfully")
    print("Functions imported successfully")

except Exception as e:
    print(f"Import failed: {e}")

## Test NewsAPI Connection

In [ ]:
try:
    articles = get_news_articles("Tesla")

    print("Articles Found:", len(articles))

    if articles:
        print("\nFirst Article:")
        print(articles[0]["title"])

except Exception as e:
    print("Error:", e)

## Test summarize_articles()

In [ ]:
try:
    summary = summarize_articles(articles)

    print("Summary:\n")
    print(summary)

except Exception as e:
    print("Error:", e)

## Test get_summary()

In [ ]:
try:
    summary = get_summary("Tesla")

    print(summary)

except Exception as e:
    print("Error:", e)

## Test get_chain()

In [ ]:
try:
    chain = get_chain("Groq (LLaMA 3.3)")

    print("Chain created successfully")
    print(chain)

except Exception as e:
    print("Error:", e)

## Test Chain Directly

In [ ]:
try:
    query = "Tesla earnings Q1 2024"

    summaries = get_summary(query)

    chain = get_chain("Groq (LLaMA 3.3)")

    result = chain.invoke({
        "query": query,
        "summaries": summaries
    })

    print(result)

except Exception as e:
    print("Error:", e)

## Test analyze_news()

In [ ]:
try:
    result = analyze_news("Tesla earnings Q1 2024")

    print("LLM Analysis:\n")
    print(result)

except Exception as e:
    print("Error:", e)

In [ ]:
# testing for why chain.invoke is not working
import os
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=os.getenv("GROQ_API_KEY"),
    model_name="llama-3.1-8b-instant"
)

response = llm.invoke("Hello")
print(response.content)

In [ ]:
# Run the full chain
chain = get_chain("Groq (LLaMA 3.3)")

response = chain.invoke({
    "query": query,
    "summaries": summaries
})


## Phase 5  Testing & Validation


In [ ]:
# Cell 5  Test imports and API keys
try:
    from langchain_config import chain, analyze_news, get_summary, get_news_articles, summarize_articles
    print("langchain_config imported successfully")
except Exception as e:
    print(f" Import failed: {e}")

from dotenv import load_dotenv
import os
load_dotenv()

keys = {
    "GROQ_API_KEY" : os.getenv("GROQ_API_KEY"),
    "NEWSAPI_KEY"  : os.getenv("NEWSAPI_KEY"),
}

print("\n=== API KEY STATUS ===")
for name, val in keys.items():
    if val:
        print(f"{name} loaded  starts with: {val[:8]}...")
    else:
        print(f"{name} NOT FOUND")

In [ ]:
# Cell 6  Test NewsAPI returns articles
try:
    articles = get_news_articles("Tesla")

    assert isinstance(articles, list),   "Result is not a list"
    assert len(articles) > 0,            "No articles returned"
    assert 'title' in articles[0],       "Article missing title"
    assert 'description' in articles[0], "Article missing description"

    print(f"   NewsAPI working  {len(articles)} articles fetched")
    print(f"   First article : {articles[0]['title']}")
    print(f"   Source        : {articles[0]['source']['name']}")

except AssertionError as e:
    print(str(e))
except Exception as e:
    print(f" NewsAPI error: {e}")

In [ ]:
# Cell 7  Test summarize_articles with edge cases
sample_articles = [
    {'title': 'Tesla misses earnings',      'description': 'Tesla reported lower than expected revenue in Q1 2024'},
    {'title': 'Elon Musk comments on growth','description': 'CEO Elon Musk said the company expects recovery in H2 2024'},
    {'title': 'No description article',     'description': None}  # should be skipped
]

result = summarize_articles(sample_articles)
print("=== SUMMARIZE TEST ===")
print(result)
print()

assert "Tesla misses earnings" in result,  "First article missing"
assert "Elon Musk" in result,              "Second article missing"
assert "No description" not in result,     " None article should be skipped"

print("summarize_articles works correctly")
print("None descriptions correctly skipped")

In [ ]:
# Cell 8  Test LLM chain gives a response
try:
    test_query = "Apple stock news"
    test_summaries = get_summary(test_query)

    assert len(test_summaries) > 0, " No summaries returned"
    print(f" Summaries fetched  {len(test_summaries)} characters\n")

    #  Use analyze_news  correct modern approach
    response = analyze_news(test_query)

    assert len(response) > 50, "LLM response too short"

    print("LLM chain working correctly")
    print(f" Response length: {len(response)} characters\n")
    print("=== SAMPLE RESPONSE ===")
    print(response[:300], "...")

except Exception as e:
    print(f"LLM chain error: {e}")

In [ ]:
# Cell 9  Full pipeline validation summary
print("=" * 50)
print("   PHASE 5  FULL PIPELINE VALIDATION")
print("=" * 50)

tests = {
    "Import langchain_config" : False,
    "API keys loaded"         : False,
    "NewsAPI fetch works"     : False,
    "Summarizer works"        : False,
    "LLM chain responds"      : False,
}

try:
    import importlib, langchain_config
    importlib.reload(langchain_config)
    from langchain_config import chain, get_summary, analyze_news
    tests["Import langchain_config"] = True

    import os
    tests["API keys loaded"] = bool(os.getenv("GROQ_API_KEY") or os.getenv("GEMINI_API_KEY"))

    s = get_summary("Microsoft")
    tests["NewsAPI fetch works"] = len(s) > 0

    tests["Summarizer works"] = "- " in s

    # Fixed  use analyze_news instead of llm_chain.run()
    r = analyze_news("Microsoft")
    tests["LLM chain responds"] = len(r) > 50

except Exception as e:
    print(f"Error during test: {e}")

print()
all_passed = True
for test_name, passed in tests.items():
    status = " PASS" if passed else "FAIL"
    print(f"  {status}    {test_name}")
    if not passed:
        all_passed = False

print()
if all_passed:
    print("ALL TESTS PASSED  Ready to run app.py!")
else:
    print("  Some tests failed  fix errors above before running app.py")